# Charger et mettre en forme les données à partir d'un fichier csv

In [2]:
import pandas 

noms = ['seismic','seismoacoustic','shift','genergy','gpuls','gdenergy','gdpuls','ghazard','nbumps','nbumps2','nbumps3', 'nbumps4','nbumps5','nbumps6','nbumps7','nbumps89','energy','maxenergy','class']
dataframe = pandas.read_csv('seismic-bumps.csv', names = noms)
dataframe

,seismic,seismoacoustic,shift,genergy,gpuls,gdenergy,gdpuls,ghazard,nbumps,nbumps2,nbumps3,nbumps4,nbumps5,nbumps6,nbumps7,nbumps89,energy,maxenergy,class
0,a,a,N,15180,48,-72,-72,a,0,0,0,0,0,0,0,0,0,0,0
1,a,a,N,14720,33,-70,-79,a,1,0,1,0,0,0,0,0,2000,2000,0
2,a,a,N,8050,30,-81,-78,a,0,0,0,0,0,0,0,0,0,0,0
3,a,a,N,28820,171,-23,40,a,1,0,1,0,0,0,0,0,3000,3000,0
4,a,a,N,12640,57,-63,-52,a,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2579,b,a,W,81410,785,432,151,b,0,0,0,0,0,0,0,0,0,0,0
2580,b,a,W,42110,555,213,118,a,0,0,0,0,0,0,0,0,0,0,0
2581,b,a,W,26960,540,101,112,a,0,0,0,0,0,0,0,0,0,0,0
2582,a,a,W,16130,322,2,2,a,0,0,0,0,0,0,0,0,0,0,0


In [3]:
donnees = dataframe.values
d = donnees[:, 0:18]
t = donnees[:, 18]

# Transformer les attributs catégoriels en un ensemble de variables binaires

In [4]:
import numpy as np

def binarisation(d, indice, valeurs):
    B = np.zeros((len(d), len(valeurs)))
    for i in range(len(d)):
        for v in range(len(valeurs)):
            if(d[i,indice] == valeurs[v]):
                B[i,v] = 1
    return B 


A0 = binarisation(d, 0,['a','b','c','d'])
A1 = binarisation(d, 1,['a','b','c','d'])
A2 = binarisation(d, 2,['W','N'])
A7 = binarisation(d, 7,['a','b','c','d'])

X = np.concatenate((A0, A1, A2, d[:,3:7],A7, d[:,8:18]), axis=1)
Y = t.astype('int')

In [5]:
pandas.DataFrame(data = X)

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,0,0,0,0
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1,0,1,0,0,0,0,0,2000,2000
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,0,0,0,0
3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1,0,1,0,0,0,0,0,3000,3000
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2579,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2580,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2581,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2582,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0


# Données deséquilibrées

In [13]:
len(X[Y == 0]), len(X[Y == 1])

(2414, 170)

# Ajustement du modèle baseline où l'on prédit toujours la classe 0

In [14]:
len(X[Y == 0])/ (len(X[Y == 1]) + len(X[Y == 0]))

0.9342105263157895

# Apprentissage du modèle

In [15]:
import random 
from sklearn.model_selection import train_test_split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.3, random_state = random.seed()) 

In [16]:
from sklearn.ensemble import RandomForestClassifier
modele = RandomForestClassifier(n_estimators = 10)
modele.fit(X_train, Y_train)
Y_predit = modele.predict(X_test)

In [17]:
from sklearn.metrics import accuracy_score
accuracy_score(Y_test, Y_predit)  

0.9342783505154639

In [21]:
from sklearn.metrics import confusion_matrix
confusion_matrix(Y_test, Y_predit)
# model has poor detection rate for class 1 from 53 it predicts only 3 correctly. This is due to the fact that the dataset is highly imbalanced.

array([[722,   1],
       [ 50,   3]])

In [19]:
from sklearn.metrics import f1_score
f1_score(Y_test, Y_predit)

0.10526315789473684

# Sous-échantillonnage de la classe majoritaire et sur-pondération

In [20]:
nbs = 170
from sklearn.utils import resample
echantillon_majorite = resample(X[Y == 0], 
                                 replace = False,    # echantillon sans remise
                                 n_samples = nbs,    # pour en avoir autant que dans la classe minoritaire
                                 random_state = random.seed()) 
# On combine la classe minoritaire avec le sous-echantillon de la classe majoritaire
X1 = np.concatenate((echantillon_majorite, X[Y == 1]), axis = 0)
Y1 = np.concatenate((np.zeros(nbs), np.ones(170)), axis = 0)

In [ ]:
X1_train, X1_test, Y1_train, Y1_test = train_test_split(X1, Y1, test_size = 0.3, random_state = random.seed()) 
    
modele.fit(X1_train, Y1_train)
Y1_predit = modele.predict(X1_test)

In [ ]:
f1_score(Y1_test, Y1_predit)

In [ ]:
confusion_matrix(Y1_test, Y1_predit)

# Surpondération de la classe sous-échantillonnée

In [ ]:
poids = (2414  / nbs)
p = (1-Y1_test) * poids
p = p + Y1_test
f1_score(Y1_test, Y1_predit, sample_weight = p)

In [ ]:
confusion_matrix(Y1_test, Y1_predit, sample_weight = p)